In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
load_dotenv()
from google import genai
from google.genai import types
gemini_client = genai.Client()  


In [10]:
import json
user_prompt = json.dumps(doc)
prompt = f"""{data_gen_instructions}\n\nFAQ record:\n{user_prompt}"""

In [11]:
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Questions,
    ),
)

In [12]:
result = Questions.model_validate_json(response.text)
result.questions

['Is it too late to register if I only just found out about this?',
 'Can I still sign up for the course now even though it has already started?',
 'What happens if I join the course late regarding the final certification?',
 'Is there still a chance to get the certificate if I start the lessons today?',
 'Do I need to worry about being unable to finish the project if I enroll at this point?']

In [13]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    gemini_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it too late to jump into the course if I just found it today?', 'Will I be able to get a certificate if I start the course right now?', 'Are new students still allowed to sign up for this cohort?', 'What do I need to do to qualify for certification if I joined late?', 'Is there a deadline for project submissions if I want to earn the certificate?']


In [16]:
usage

GenerateContentResponseUsageMetadata(
  candidates_token_count=83,
  prompt_token_count=264,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=264
    ),
  ],
  total_token_count=347
)

In [17]:
from evaluation_utils import calc_price

In [18]:
calc_price(usage)

{'input_cost': 2.6400000000000005e-05,
 'output_cost': 3.32e-05,
 'total_cost': 5.9600000000000005e-05}

In [19]:
records = [
    {"question": question, "document": doc["id"]}
    for question in result.questions
]
records

[{'question': 'Is it too late to jump into the course if I just found it today?',
  'document': '74eb249bbf'},
 {'question': 'Will I be able to get a certificate if I start the course right now?',
  'document': '74eb249bbf'},
 {'question': 'Are new students still allowed to sign up for this cohort?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for certification if I joined late?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for project submissions if I want to earn the certificate?',
  'document': '74eb249bbf'}]

In [20]:
import pandas as pd

In [21]:
pd.DataFrame(records)

,question,document
0,Is it too late to jump into the course if I ju...,74eb249bbf
1,Will I be able to get a certificate if I start...,74eb249bbf
2,Are new students still allowed to sign up for ...,74eb249bbf
3,What do I need to do to qualify for certificat...,74eb249bbf
4,Is there a deadline for project submissions if...,74eb249bbf


In [22]:
from evaluation_utils import llm_structured_retry

In [23]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        gemini_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [24]:
generate_ground_truth(doc)

([{'question': 'Is it too late to jump into the course if I just found it?',
   'document': '74eb249bbf'},
  {'question': 'Can I register for this program even though it already started?',
   'document': '74eb249bbf'},
  {'question': 'Will I be able to get a certificate if I sign up right now?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to qualify for certification if I start late?',
   'document': '74eb249bbf'},
  {'question': 'Are there any deadlines I should know about if I join the class today?',
   'document': '74eb249bbf'}],
 GenerateContentResponseUsageMetadata(
   candidates_token_count=99,
   prompt_token_count=264,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=264
     ),
   ],
   total_token_count=363
 ))

In [25]:
from tqdm.auto import tqdm
import time


ground_truth = []
usages = []

for doc in tqdm(documents):
    results, usage = generate_ground_truth(doc)
    ground_truth.extend(results)
    usages.append(usage)
    time.sleep(4.5)  # Add a delay of 4.5 seconds between requests to avoid rate limiting

  0%|          | 0/118 [00:00<?, ?it/s]

In [ ]:
# i skipped the parallel processing because it was causing rate limiting issues. Instead, I used the usaual sequential processing with a delay between requests to avoid hitting the rate limit.

In [26]:
len(ground_truth)

590

In [27]:
ground_truth[10]

{'question': 'Where can I find the direct Zoom link to join the live workshop sessions?',
 'document': '489dd1c9d9'}

In [28]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.009893800000000001

In [29]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.009893800000000001

In [30]:
df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [35]:
len(df_ground_truth)

590